In [6]:
import os
import cv2
import numpy as np
from ultralytics import YOLO

# Путь к папке datasets
datasets_dir = "datasets/peper"

# Пути к изображениям и аннотациям
images_train_dir = os.path.join(datasets_dir, "images", "train")
images_val_dir = os.path.join(datasets_dir, "images", "val")
labels_train_dir = os.path.join(datasets_dir, "labels", "train")
labels_val_dir = os.path.join(datasets_dir, "labels", "val")

# Проверка существования папок
os.makedirs(images_train_dir, exist_ok=True)
os.makedirs(images_val_dir, exist_ok=True)
os.makedirs(labels_train_dir, exist_ok=True)
os.makedirs(labels_val_dir, exist_ok=True)

# Создаем конфигурационный файл для YOLOv8
yolo_config = f"""
path: {datasets_dir}  # Путь к датасету
train: images/train   # Папка с обучающими изображениями
val: images/val       # Папка с валидационными изображениями
nc: 1                 # Количество классов (1 для листа бумаги)
names: ['paper']      # Имена классов
"""

# Сохраняем конфигурационный файл
config_path = os.path.join(datasets_dir, "yolov8_config.yaml")
with open(config_path, "w") as f:
    f.write(yolo_config)

# Загрузка предобученной модели YOLOv8
model = YOLO("yolov8n.pt")

# Обучение модели
results = model.train(data=config_path, epochs=50, imgsz=640)

# Функция для выравнивания изображения на основе bounding box
def align_image(image, bbox):
    """
    Выравнивает изображение на основе bounding box.

    Параметры:
        image (np.array): Исходное изображение.
        bbox (list): Координаты bounding box [x_min, y_min, x_max, y_max].

    Возвращает:
        np.array: Выровненное изображение.
    """
    x_min, y_min, x_max, y_max = bbox
    corners = np.array([
        [x_min, y_min],
        [x_max, y_min],
        [x_max, y_max],
        [x_min, y_max]
    ], dtype="float32")

    height, width = image.shape[:2]
    dst = np.array([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1]
    ], dtype="float32")

    M = cv2.getPerspectiveTransform(corners, dst)
    aligned_image = cv2.warpPerspective(image, M, (width, height))
    return aligned_image

# Пример использования обученной модели
input_folder = "peper/extracted_frames"  # Папка с исходными изображениями
output_folder = "peper/aligned_images"   # Папка для сохранения выровненных изображений
os.makedirs(output_folder, exist_ok=True)

# Получаем список всех изображений в папке
image_files = [f for f in os.listdir(input_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]

# Обрабатываем каждое изображение
for image_file in image_files:
    input_path = os.path.join(input_folder, image_file)
    output_path = os.path.join(output_folder, f"aligned_{image_file}")

    # Загружаем и обрабатываем изображение
    try:
        # Загрузка изображения
        image = cv2.imread(input_path)
        if image is None:
            print(f"Ошибка: Не удалось загрузить изображение {input_path}.")
            continue

        # Детекция листа бумаги
        results = model(image)
        bbox = results[0].boxes.xyxy[0].cpu().numpy()  # Получаем bounding box

        # Выравнивание изображения
        aligned_image = align_image(image, bbox)

        # Сохранение результата
        cv2.imwrite(output_path, aligned_image)
        print(f"Выровненное изображение сохранено как {output_path}.")

        # Показываем результат (если установлен matplotlib)
        try:
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(1, 2, figsize=(10, 5))
            ax[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            ax[0].set_title("Исходное изображение")
            ax[0].axis('off')

            ax[1].imshow(cv2.cvtColor(aligned_image, cv2.COLOR_BGR2RGB))
            ax[1].set_title("Выровненное изображение")
            ax[1].axis('off')

            plt.show()
        except ImportError:
            print("Для визуализации установите matplotlib: pip install matplotlib")

    except Exception as e:
        print(f"Ошибка при обработке изображения {input_path}: {e}")

New https://pypi.org/project/ultralytics/8.3.65 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.64  Python-3.12.7 torch-2.5.1+cpu CPU (13th Gen Intel Core(TM) i7-13700H)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=datasets/peper\yolov8_config.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train3, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=Fals

RuntimeError: Dataset 'datasets/peper/yolov8_config.yaml' error  
Dataset 'datasets/peper/yolov8_config.yaml' images not found , missing path 'C:\Users\\Desktop\\-\\datasets\datasets\peper\images\val'
Note dataset download directory is 'C:\Users\\Desktop\\-\\datasets'. You can update this in 'C:\Users\\AppData\Roaming\Ultralytics\settings.json'